In [3]:
import pandas as pd
import numpy as np
import re

# ==========================================
# 1. VERİ TEMİZLEME AŞAMASI
# ==========================================
df_aciklama = pd.read_excel("../data/raw/dogo_iade_aciklamali.xlsx")

# Sütun adlarındaki gizli boşlukları temizle
df_aciklama.columns = df_aciklama.columns.str.strip()

# Görünmez karakterleri temizle (_x000D_, \n vb.)
df_aciklama = df_aciklama.replace(to_replace=[r'_x000D_', r'\r', r'\n'], value=' ', regex=True)

# Sipariş No'yu string (metin) olarak kabul et ve sağ/sol boşluklarını sil
df_aciklama['Sipariş No'] = df_aciklama['Sipariş No'].astype(str).str.strip()

# Sipariş No'su 'nan', '0', veya tamamen boş ('') olanları filtrele
istenmeyen_degerler = ['nan', '0', '', 'None']
df_temiz = df_aciklama[~df_aciklama['Sipariş No'].isin(istenmeyen_degerler)].copy()

print(f"Temizlik öncesi satır sayısı: {len(df_aciklama)}")
print(f"Temizlik sonrası sağlam satır sayısı: {len(df_temiz)}")


# ==========================================
# 2. NLP VE KATEGORİ ÇIKARIMI AŞAMASI
# ==========================================
def kategorileri_cikar(urun_adi):
    if pd.isna(urun_adi):
         return "Bilinmiyor", "Bilinmiyor"
    
    urun_adi_lower = str(urun_adi).lower()
    
    kategori_sozlugu = {
        "Ayakkabı": ["sneakers", "günlük ayakkabı", "sandalet", "terlik", "bot", "topuklu", "outdoor", "loafer", "babet"],
        "Çanta": ["omuz çantası", "sırt çantası", "seyahat çantası", "el çantası", "tote bag", "çanta"],
        "Giyim": ["çorap", "t-shirt", "tshirt", "tişört", "giyim"],
        "Aksesuar": ["cüzdan", "pasaport", "fular", "bandana", "aksesuar"]
    }
    
    for ana_kategori, alt_kategoriler in kategori_sozlugu.items():
        for alt_kat in alt_kategoriler:
            if re.search(r'\b' + re.escape(alt_kat) + r'\b', urun_adi_lower):
                return ana_kategori, alt_kat.title()
                
    return "Diğer", "Diğer"

# Series yerine Tuple dönüyoruz ve zip(*) ile Pandas'ın en güvenli atama yöntemini kullanıyoruz.
df_temiz['Ana Kategori'], df_temiz['Alt Kategori'] = zip(*df_temiz['Ürün Adı'].apply(kategorileri_cikar))

# Sonucu Göster
display(df_temiz[['Sipariş No', 'Ürün Adı', 'Ana Kategori', 'Alt Kategori']].head(10))

Temizlik öncesi satır sayısı: 498
Temizlik sonrası sağlam satır sayısı: 498


,Sipariş No,Ürün Adı,Ana Kategori,Alt Kategori
0,TS310798462,Kadın Vegan Deri Turkuaz Terlik - Feeling Mood...,Ayakkabı,Terlik
1,TS310798462,Kadın Vegan Deri Beyaz Sneakers - Pink Paradis...,Ayakkabı,Sneakers
2,TS020898575,Kadın Vegan Deri Gri Uzun Bot - Bitter Sweet W...,Ayakkabı,Bot
3,TS020898558,Kadın Vegan Deri Çok Renkli Terlik - Burano Is...,Ayakkabı,Terlik
4,TS020898558,Kadın Vegan Deri Bej Bağcıklı Sandalet - Stork...,Ayakkabı,Sandalet
5,TS260798257,Kadın Vegan Deri Beyaz Sneakers - Vincent van ...,Ayakkabı,Sneakers
6,TS260798257,Kadın Vegan Deri Bej Kalın Taban Sandalet - Mo...,Ayakkabı,Sandalet
7,TS310798474,Unisex Vegan Gri Sırt Çantası - Warner Bros To...,Çanta,Sırt Çantası
8,TS240798164,Kadın Vegan Deri Yeşil Loafer - I Do What I Wa...,Ayakkabı,Loafer
9,TS200797970,Kadın Vegan Deri Gri Sneakers - Astronomy Map ...,Ayakkabı,Sneakers


In [4]:
# Türkçe karakterlerin Excel'de bozulmaması için utf-8-sig kullanıyoruz.
# Excel'de sütunların düzgün ayrılması için ayırıcıyı noktalı virgül (;) yapıyoruz.
dosya_yolu = "../data/processed/01_temizlenmis_aciklamali_iade.csv"

df_temiz.to_csv(dosya_yolu, index=False, encoding='utf-8-sig', sep=';')

print(f"Temizlenmiş veri başarıyla kaydedildi: {dosya_yolu}")

Temizlenmiş veri başarıyla kaydedildi: ../data/processed/01_temizlenmis_aciklamali_iade.csv
